# Fare Explainer Notebook
This notebook demonstrates data loading, feature engineering, training an XGBoost GBM, SHAP explainability, residual analysis, and a small query function to evaluate if a specific flight is inflated.

In [ ]:
# 1. Imports and helpers
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import shap
# ensure package import path
sys.path.append(str(Path('..').resolve()))
from src.utils import feature_engineer
sns.set(style='whitegrid')

In [ ]:
# 2. Load data and run feature engineering
data_path = Path('..') / 'data' / 'airline_ticket_dataset_csv.csv'
df = pd.read_csv(data_path, dtype=str)
df = feature_engineer(df)
print('rows,cols', df.shape)
df.head()

## Feature selection and temporal encoding
We create `fare_per_mile` as a derived unit, a `competitive_pressure` index, and `market_concentration`. We also one-hot `Year` & `quarter` to capture seasonality.

In [ ]:
# select features and encode time
candidates = ['nkm','fare_per_km','fare_per_mile','passengers_clean','orig_total','dest_total','hub_ratio','origin_is_hub','hub_intensity_orig','hub_intensity_dest','market_concentration','competitive_pressure','large_ms','lf_ms','carrier_lg_freq','carrier_low_freq']
features = [c for c in candidates if c in df.columns]
for c in ['Year','quarter']:
    if c in df.columns:
        df[c] = df[c].astype(str)
df = pd.get_dummies(df, columns=[c for c in ['Year','quarter'] if c in df.columns], drop_first=True)
features += [c for c in df.columns if c.startswith('Year_') or c.startswith('quarter_')]
print('Using features:', features)

In [ ]:
# 4. Prepare target and split
df['fare_clean'] = pd.to_numeric(df['fare_clean'], errors='coerce')
df = df.dropna(subset=['fare_clean','nsmiles_clean'])
if 'fare_per_mile' not in df.columns:
    df['fare_per_mile'] = df['fare_clean'] / df['nsmiles_clean'].replace({0:np.nan})
X = df[features].fillna(df[features].median())
y = df['fare_per_mile'].fillna(df['fare_per_mile'].median())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3169)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=3169)
print('train size', X_tr.shape, 'val size', X_val.shape, 'test size', X_test.shape)

## Train GBM (XGBoost)
We train a GBM to predict `fare_per_mile`. Predicted fare = predicted_per_mile * distance.

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, tree_method='hist', random_state=3169)
callbacks = [xgb.callback.EarlyStopping(rounds=50, save_best=True)]
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mae', callbacks=callbacks)
preds = model.predict(X_test)
print('Test MAE (per-mile):', mean_absolute_error(y_test, preds))

## SHAP explainability
Compute SHAP values to show feature importance and per-row explanations.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(10,6))
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()

## Residual analysis (overpriced vs. high-value corridors)
Residual = Actual - Predicted. Positive residuals indicate overpriced routes.

In [ ]:
full_preds_per_mile = model.predict(X)
df['pred_per_mile'] = full_preds_per_mile
df['predicted_fare'] = df['pred_per_mile'] * df['nsmiles_clean']
df['residual'] = df['fare_clean'] - df['predicted_fare']
# flag inflated if residual pct > 0.15
df['pct_diff'] = df['residual'] / df['predicted_fare'].replace({0:np.nan})
df['inflated'] = df['pct_diff'] > 0.15
plt.figure(figsize=(8,6))
sns.scatterplot(x='predicted_fare', y='fare_clean', hue='inflated', data=df.sample(min(2000,len(df))), alpha=0.7)
plt.plot([0, df['predicted_fare'].max()],[0, df['predicted_fare'].max()], color='k', linestyle='--')
plt.xlabel('Predicted Fare')
plt.ylabel('Actual Fare')
plt.title('Actual vs Predicted Fare (sample)')
plt.show()

## Simple query: evaluate a city-pair and suggest alternatives

In [ ]:
def evaluate_route(origin, dest, df=df, model=model, features=features, top_k=3):
    # find row matching origin/dest (first match)
    candidate = df[(df['city1']==origin)&(df['city2']==dest)]
    if candidate.empty:
        return f'No route row found for {origin} -> {dest}'
    row = candidate.iloc[0]
    x = row[features].values.reshape(1,-1)
    pred_per_mile = model.predict(x)[0]
    pred_fare = pred_per_mile * row['nsmiles_clean']
    actual = row['fare_clean']
    residual = actual - pred_fare
    value_rating = 'Fair' if abs(residual / pred_fare) < 0.05 else ('Overpriced' if residual>0 else 'Underpriced')
    origin_df = df[df['city1']==origin].copy()
    origin_df['pred_per_mile'] = model.predict(origin_df[features])
    origin_df['predicted_fare'] = origin_df['pred_per_mile'] * origin_df['nsmiles_clean']
    origin_df['cost_per_km'] = origin_df['predicted_fare'] / origin_df['nkm'].replace({0:np.nan})
    alternatives = origin_df.sort_values('cost_per_km')[['city2','predicted_fare','cost_per_km']].head(top_k)
    return {
        'origin': origin,
        'dest': dest,
        'actual_fare': actual,
        'predicted_fare': pred_fare,
        'residual': residual,
        'pct_diff': residual / pred_fare if pred_fare!=0 else np.nan,
        'value_rating': value_rating,
        'alternatives': alternatives.to_dict(orient='records')
    }

In [ ]:
# Example: evaluate a sample route (change names as in your CSV)
sample = df[['city1','city2']].dropna().iloc[0]
print(evaluate_route(sample['city1'], sample['city2']))